# Week 8 Mini Assignment — From Agent Trace to Safety Case

**ESP3201 · AI Agents in Robotics Engineering**

This is an offline, CPU-only Colab notebook: it needs no API key, package install, or repository checkout. You will audit two recorded agent runs on one task: make a navigation controller respond safely when camera data becomes stale.

> **Rule:** an agent's plan, patch, self-critique, and passing unit test are candidates. An observation supports only the claim it actually measured.

Your submission is an evidence ledger plus an acceptance decision. Do not write safety-critical code from scratch.

## Learning goals

By the end you should be able to:

- audit an agent's action/observation trace rather than trust its final narration;
- distinguish reflection from external feedback;
- match component, integration, scenario, and hardware checks to claims; and
- make a bounded accept / revise / reject decision with a traceable disclosure.

In [ ]:
# No imports or downloads required. These are immutable records from a teaching harness.
TASK = 'If no camera frame arrives for more than 200 ms, stop unsafe motion.'

TOOL_CONTRACTS = {
    'read_file': {'permission': 'read-only repository', 'observation': 'versioned file content'},
    'run_unit': {'permission': 'sandboxed test process', 'observation': 'local function result'},
    'run_scenario': {'permission': 'simulated driver + queue', 'observation': 'queue, driver and time trace'},
    'propose_patch': {'permission': 'draft only; human review required before merge', 'observation': 'candidate diff'},
}

RUN_V1 = [
    {'id': 'v1.01', 'kind': 'OBS', 'text': 'requirements.md: stale threshold = 200 ms; driver may replay cached commands; queue exists.'},
    {'id': 'v1.02', 'kind': 'ACTION', 'text': 'read_file(safety/stale_guard.py)'},
    {'id': 'v1.03', 'kind': 'OBS', 'text': 'frame_stamp is produced by scheduler_monotonic_now().'},
    {'id': 'v1.04', 'kind': 'THOUGHT', 'text': 'Suppressing a future publish should stop the robot.'},
    {'id': 'v1.05', 'kind': 'ACTION', 'text': 'propose_patch: if time.time() - frame_stamp > 0.2: return'},
    {'id': 'v1.06', 'kind': 'ACTION', 'text': 'run_unit(test_skips_when_camera_is_stale)'},
    {'id': 'v1.07', 'kind': 'OBS', 'text': 'PASS: publisher.messages == [] for one stale timestamp.'},
    {'id': 'v1.08', 'kind': 'FINAL', 'text': 'This patch guarantees the robot will not move under stale perception.'},
]

RUN_V2 = [
    {'id': 'v2.01', 'kind': 'OBS', 'text': 'scenario: safety suppresses new publish; driver executes cached vx=0.4; queue depth=2.'},
    {'id': 'v2.02', 'kind': 'THOUGHT', 'text': 'The unit test proved only a local publish branch, not driver or queue behaviour.'},
    {'id': 'v2.03', 'kind': 'ACTION', 'text': 'read_file(driver/queue_api.md)'},
    {'id': 'v2.04', 'kind': 'OBS', 'text': 'clear_pending_motion() empties queue; send_zero() is delivered to driver in simulation.'},
    {'id': 'v2.05', 'kind': 'ACTION', 'text': 'propose_patch: use monotonic arrival time; clear queue; send zero; latch stale state.'},
    {'id': 'v2.06', 'kind': 'ACTION', 'text': 'run_scenario(stale_frame_with_queued_motion)'},
    {'id': 'v2.07', 'kind': 'OBS', 'text': 'PASS in simulation: queue=0, driver received zero command, elapsed=120 ms.'},
    {'id': 'v2.08', 'kind': 'FINAL', 'text': 'Candidate passes this simulated queue scenario. Hardware timing and network conditions remain unverified.'},
]

def show_trace(trace):
    for row in trace:
        print(f"{row['id']:>5}  {row['kind']:<7}  {row['text']}")

print('Task:', TASK)
print('Available tool contracts:')
for name, contract in TOOL_CONTRACTS.items():
    print(f"- {name}: {contract['permission']} → {contract['observation']}")

## Part A — Audit the first run

Read the trace. Identify three distinct problems:

1. a clock-domain problem;
2. a system-boundary problem; and
3. an overclaimed conclusion.

For each, cite at least one exact trace ID.

In [ ]:
show_trace(RUN_V1)

### Part A worksheet

Fill this in your submitted document.

| Finding | Trace ID(s) | Why it matters | Smallest adequate verifier |
| --- | --- | --- | --- |
| Mixed clock domain | | | |
| Cached / queued motion | | | |
| Unsupported safety claim | | | |

A thought is not evidence. A unit-test observation supports only its local assertion.

## Part B — Reflection or external feedback?

The second run follows a scenario observation. Compare it with the first. Which line introduced **new external information**? Which line is merely a better hypothesis?

In [ ]:
show_trace(RUN_V2)

def observations(trace):
    return [row for row in trace if row['kind'] == 'OBS']

print('\nExternal observations in v2:')
for row in observations(RUN_V2):
    print('-', row['id'], row['text'])

## Part C — Build an evidence ledger

Complete four rows. Each row must name a claim, a concrete oracle/observation, the verification layer, and a remaining limit.

- `component`: local code or API contract
- `integration`: message/interface path
- `scenario`: multi-component state, queue, recovery, or timing scenario
- `hardware`: physical timing or deployment condition

Do **not** use “add more tests” as an oracle. Name what will be observed and under what condition.

In [ ]:
# Replace every TODO before submitting. The validator checks completeness, not your engineering judgement.
evidence_ledger = [
    {'claim': 'TODO', 'oracle': 'TODO', 'layer': 'TODO', 'limit': 'TODO', 'trace_ids': ['TODO']},
    {'claim': 'TODO', 'oracle': 'TODO', 'layer': 'TODO', 'limit': 'TODO', 'trace_ids': ['TODO']},
    {'claim': 'TODO', 'oracle': 'TODO', 'layer': 'TODO', 'limit': 'TODO', 'trace_ids': ['TODO']},
    {'claim': 'TODO', 'oracle': 'TODO', 'layer': 'TODO', 'limit': 'TODO', 'trace_ids': ['TODO']},
]

VALID_LAYERS = {'component', 'integration', 'scenario', 'hardware'}
ALL_IDS = {row['id'] for row in RUN_V1 + RUN_V2}

def validate_ledger(rows):
    assert len(rows) == 4, 'Use exactly four evidence-ledger rows.'
    for i, row in enumerate(rows, 1):
        for field in ('claim', 'oracle', 'layer', 'limit', 'trace_ids'):
            assert row.get(field) and row[field] != 'TODO', f'Row {i}: complete {field}.'
        assert row['layer'] in VALID_LAYERS, f"Row {i}: choose one of {sorted(VALID_LAYERS)}."
        assert set(row['trace_ids']).issubset(ALL_IDS), f'Row {i}: use real trace IDs.'
    print('Structure complete. Now check that your oracle actually matches your claim.')

# Uncomment after editing:
# validate_ledger(evidence_ledger)

## Part D — Acceptance decision and regression proposal

Choose **accept**, **accept with revisions**, or **reject** for the v2 candidate. Your decision must:

- state the strongest claim the evidence supports;
- name one claim it does not support;
- cite at least two trace IDs; and
- specify one future regression scenario: inputs, expected observation, and verification layer.

Finally disclose any AI assistance: what it helped draft, what you verified manually, and one suggestion you rejected or bounded.

## Submission checklist

Submit a PDF or Markdown document containing:

1. the completed Part A finding table;
2. the Part B distinction between reflection and external feedback;
3. the four-row evidence ledger;
4. an evidence-bounded acceptance decision and regression proposal; and
5. an AI-agent usage disclosure.

**Compute note:** this notebook is fully offline and CPU-only. No private API key, paid model, or GPU is required.